# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue across all orders is ${total_revenue:,.2f}.")
print(f"The total number of units sold across all orders is {total_units:,.0f}.")

The total revenue across all orders is $8,520.00.
The total number of units sold across all orders is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
revenue_by_category['share_of_total'] = (revenue_by_category['revenue'] / total_revenue) * 100

print("This shows by category, from highest to lowest, including their share of the total revenue:")
display(revenue_by_category.style.format({'revenue': '${:,.2f}', 'share_of_total': '{:,.2f}%'}))


This shows by category, from highest to lowest, including their share of the total revenue:


,category,revenue,share_of_total
0,Food,"$4,293.00",50.39%
1,Merch,"$1,771.50",20.79%
2,Drink,"$1,554.00",18.24%
3,RainGear,$901.50,10.58%


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_summary = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    order_count=('vendor_id', 'count')
).sort_values(by='avg_revenue', ascending=False)

highest_avg_vendor = vendor_summary.iloc[0]

display(vendor_summary.style.format({'avg_revenue': '${:,.2f}'}))
print(f"\nThe vendor with the highest average order revenue is {highest_avg_vendor.name}, averaging ${highest_avg_vendor['avg_revenue']:.2f} per order for {int(highest_avg_vendor['order_count'])} orders.")

,avg_revenue,order_count
vendor_id,,
V-01,$22.60,94
V-18,$21.75,108
V-05,$20.58,93
V-10,$20.31,105



The vendor with the highest average order revenue is V-01, averaging $22.60 per order for 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue_share = revenue_by_category[revenue_by_category['category'] == 'Merch']['share_of_total'].iloc[0]

print(f"The share of total revenue that comes from merch is {merch_revenue_share:.1f}%")

The share of total revenue that comes from merch is 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

merged_df = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')

#update the vendor_name for V-18
merged_df.loc[merged_df['vendor_id'] == 'V-18', 'vendor_name'] = 'Unknown vendor 1'

initial_row_count = len(df)
merged_row_count = len(merged_df)
initial_total_revenue = df['revenue'].sum()
merged_total_revenue = merged_df['revenue'].sum()

print(f"Initial row count: {initial_row_count}")
print(f"Merged row count: {merged_row_count}")
print(f"Initial total revenue: ${initial_total_revenue:.2f}")
print(f"Merged total revenue: ${merged_total_revenue:.2f}")

unmatched_vendor_id = merged_df[merged_df['vendor_name'].isna()]['vendor_id'].unique()
print(f"\nNot found in the vendor_names lookup: {unmatched_vendor_id[0] if len(unmatched_vendor_id) > 0 else 'None'}")

display(merged_df.head())

Initial row count: 400
Merged row count: 400
Initial total revenue: $8520.00
Merged total revenue: $8520.00

Not found in the vendor_names lookup: None


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor 1
2,V-18,Drink,3,4.5,13.5,Unknown vendor 1
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor 1


The vendor ID not found was V-18. I decided to just change the name to "Unknown vendor 1" in case we figure out the name later and we can add it without issue.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_table_revenue = pd.pivot_table(merged_df, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True, fill_value=0)
display(pivot_table_revenue.style.format('${:,.2f}'))

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,$502.50,"$1,054.50",$400.50,$175.50,"$2,133.00"
Hoos Burgers,$171.00,"$1,338.00",$373.50,$241.50,"$2,124.00"
Rotunda Tacos,$298.50,$882.00,$489.00,$244.50,"$1,914.00"
Unknown vendor 1,$582.00,"$1,018.50",$508.50,$240.00,"$2,349.00"
All,"$1,554.00","$4,293.00","$1,771.50",$901.50,"$8,520.00"


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged_df) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would advise the vendors to prioritize food because it accounts for 50.39% of total revenue. Having a great food product at a good price would allow a vendor to capture a huge share of total revenue. However, I do think that each vendor should adjust their strategy based on their strengths and weaknesses. Hoos Burgers, for example, does well in food sales, but is lagging behind Cav Merch in beverage sales, the latter of whom exceeds $500 dollars from Drinks.


b) The answer regarding for (Q6) is arguably the least trustworthy. While the pivot table provides a good overview of named vendors, the fact that 'Unknown vendor 1' (V-18) contributes to the overall revenue but is not explicitly detailed in the pivot table limits its comprehensiveness. The lack of specific breakdown for 'Unknown vendor 1' means that any conclusions drawn solely from this table might be incomplete or misleading, as the activities of a significant contributor are obscured. This probably speakers more to broadly to the weakness of unmatched data in a lookup and the priority that should be given to properly collect and clean data.